In [14]:
import torch
import os
import json
from pathlib import Path

from IPython.display import Markdown, display
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, PrinterCallback, set_seed
from trl import SFTConfig, SFTTrainer

In [3]:
# to avoid access to network (can make it faster?)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# check if we're seeing the GPU
print(torch.__version__)
print(torch.cuda.is_available())

2.14.0+cu130
True


In [4]:
print("transformers:", transformers.__version__)

transformers: 4.57.6


In [5]:
# datasets and base model are in a Volume linked to Object Storage
ROOT_DIR = Path("/Volumes/fine_tuning/fine_tuning/vol_finetuning")
DATASETS_DIR = ROOT_DIR / "datasets"
MODELS_DIR = ROOT_DIR / "models"

!ls -lah "$DATASETS_DIR"

total 0
drwxrwxrwx. 0 1000 1000    0 Sep 14 09:05 .
drwxrwxrwx. 0 1000 1000    0 Sep 14 07:53 ..
-rwxrwxrwx. 1 1000 1000  54K Sep 14 09:06 test_dataset.jsonl
-rwxrwxrwx. 1 1000 1000 282K Sep 14 09:06 train_dataset.jsonl
-rwxrwxrwx. 1 1000 1000 343K Sep 14 09:06 train_dataset_v2.jsonl
-rwxrwxrwx. 1 1000 1000  58K Sep 14 09:06 validation_dataset.jsonl
-rwxrwxrwx. 1 1000 1000  69K Sep 14 09:06 validation_dataset_v2.jsonl


In [6]:
datasets = load_dataset(
    "json",
    data_files={
        "train": f"{DATASETS_DIR}/train_dataset_v2.jsonl",
        "validation": f"{DATASETS_DIR}/validation_dataset_v2.jsonl",
    },
)

print(datasets)
print(datasets["train"][0])

DatasetDict({
    train: Dataset({
        features: ["messages"],
        num_rows: 1000
    })
    validation: Dataset({
        features: ["messages"],
        num_rows: 200
    })
})
{"messages": [{"role": "user", "content": "In the production environment, the laptop display fails after startup. The issue affects most users and blocks a critical business operation. No workaround is available."}, {"role": "assistant", "content": "{"category":"DEV-01","severity":"P1","summary":"Laptop display hardware is malfunctioning"}"}]}


In [7]:
import os

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_DIR = MODELS_DIR / "Qwen3-1.7B"
complete_file = MODEL_DIR / "config.json"

if not os.path.exists(complete_file):
    from huggingface_hub import login, snapshot_download
    
    hf_token = "INSERISCI_IL_NUOVO_TOKEN"
    login(token=hf_token, add_to_git_credential=False)

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        token=hf_token,
        max_workers=4,
    )

    open(complete_file, "w").close()

print(f"Model available in: {MODEL_DIR}")

Model available in: /Volumes/fine_tuning/fine_tuning/vol_finetuning/models/Qwen3-1.7B


In [8]:
# EDIT FOR THE LINUX HOST: base directory for checkpoints, logs, tokenizer, metadata, and LoRA adapter.
TRAINING_OUTPUT_DIR = MODELS_DIR / 'qwen3-1.7b-ticket-classification-lora'

TRAIN_FILE = DATASETS_DIR / 'train_dataset_v2.jsonl'
VALIDATION_FILE = DATASETS_DIR / 'validation_dataset_v2.jsonl'
ADAPTER_OUTPUT_DIR = TRAINING_OUTPUT_DIR / 'adapter'

for required in (MODEL_DIR / 'config.json', MODEL_DIR / 'tokenizer.json', TRAIN_FILE, VALIDATION_FILE):
    if not required.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required}')
if not (MODEL_DIR / 'model.safetensors').is_file() and not (MODEL_DIR / 'model.safetensors.index.json').is_file():
    raise FileNotFoundError(f'Missing model weights in {MODEL_DIR}.')

print(f'Model directory: {MODEL_DIR}')
print(f'Dataset directory: {DATASETS_DIR}')
print(f'LoRA output base directory: {TRAINING_OUTPUT_DIR}')

Model directory: /Volumes/fine_tuning/fine_tuning/vol_finetuning/models/Qwen3-1.7B
Dataset directory: /Volumes/fine_tuning/fine_tuning/vol_finetuning/datasets
LoRA output base directory: /Volumes/fine_tuning/fine_tuning/vol_finetuning/models/qwen3-1.7b-ticket-classification-lora


In [15]:
# helper functions
def to_prompt_completion(record: dict) -> dict:
    """Convert a ticket record to TRL prompt/completion format."""
    return {'prompt': [record['messages'][0]], 'completion': [record['messages'][1]]}


def validate_split(split_name: str, dataset) -> set[str]:
    """Validate ticket records and return ticket texts without displaying them."""
    ticket_texts = set()
    for index, record in enumerate(dataset):
        messages = record.get('messages')
        if not isinstance(messages, list) or len(messages) != 2:
            raise ValueError(f'{split_name} record {index} must contain exactly two messages.')
        if [message.get('role') for message in messages] != ['user', 'assistant']:
            raise ValueError(f'{split_name} record {index} must have user and assistant roles.')
        ticket_text = messages[0].get('content', '').strip()
        if not ticket_text or ticket_text in ticket_texts:
            raise ValueError(f'{split_name} record {index} has empty or duplicate ticket text.')
        ticket_texts.add(ticket_text)
        try:
            output = json.loads(messages[1].get('content', ''))
        except json.JSONDecodeError as error:
            raise ValueError(f'{split_name} record {index} has invalid assistant JSON.') from error
        if not isinstance(output, dict) or set(output) != {'category', 'severity', 'summary'}:
            raise ValueError(f'{split_name} record {index} must contain category, severity, and summary only.')
        if output['category'] not in ALLOWED_CATEGORIES or output['severity'] not in ALLOWED_SEVERITIES:
            raise ValueError(f'{split_name} record {index} has an unsupported label.')
        if not isinstance(output['summary'], str) or not output['summary'].strip():
            raise ValueError(f'{split_name} record {index} has an empty summary.')
    return ticket_texts

def calculate_validation_generation_metrics(model, tokenizer, validation_dataset) -> dict[str, float]:
    """Generate validation responses and return aggregate task metrics."""
    counts = {'valid': 0, 'category': 0, 'severity': 0, 'joint': 0}
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    model.eval()
    try:
        for start in range(0, len(validation_dataset), VALIDATION_GENERATION_BATCH_SIZE):
            records = validation_dataset.select(range(start, min(start + VALIDATION_GENERATION_BATCH_SIZE, len(validation_dataset))))
            prompts = [tokenizer.apply_chat_template([record['messages'][0]], tokenize=False, add_generation_prompt=True, enable_thinking=False) for record in records]
            inputs = {name: value.to(DEVICE) for name, value in tokenizer(prompts, return_tensors='pt', padding=True).items()}
            with torch.inference_mode():
                output_ids = model.generate(**inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            predictions = tokenizer.batch_decode(output_ids[:, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            for record, prediction in zip(records, predictions):
                try:
                    generated = json.loads(prediction.strip())
                    valid = isinstance(generated, dict) and set(generated) == {'category', 'severity', 'summary'}
                except json.JSONDecodeError:
                    generated, valid = {}, False
                reference = json.loads(record['messages'][1]['content'])
                category = valid and generated.get('category') == reference['category']
                severity = valid and generated.get('severity') == reference['severity']
                counts['valid'] += valid; counts['category'] += category; counts['severity'] += severity; counts['joint'] += category and severity
    finally:
        tokenizer.padding_side = original_padding_side
    total = len(validation_dataset)
    return {'category_accuracy': counts['category'] / total, 'severity_accuracy': counts['severity'] / total, 'joint_accuracy': counts['joint'] / total, 'valid_json_rate': counts['valid'] / total}

METRIC_COLUMNS = (
    "epoch",
    "training_loss",
    "validation_loss",
    "category_accuracy",
    "severity_accuracy",
    "joint_accuracy",
)


def latest_training_loss(log_history):
    return next(
        (entry["loss"] for entry in reversed(log_history) if "loss" in entry),
        float("nan"),
    )


def format_metric(value):
    try:
        return f"{float(value):.4f}"
    except (TypeError, ValueError):
        return str(value)


def print_metrics_table(rows):
    formatted_rows = [
        [format_metric(row.get(column, "")) for column in METRIC_COLUMNS]
        for row in rows
    ]

    widths = [
        max(
            len(column),
            max((len(row[index]) for row in formatted_rows), default=0),
        )
        for index, column in enumerate(METRIC_COLUMNS)
    ]

    def format_row(values):
        return " | ".join(
            value.ljust(width)
            for value, width in zip(values, widths)
        )

    print("\nMetriche per epoch")
    print(format_row(METRIC_COLUMNS))
    print("-+-".join("-" * width for width in widths))

    for row in formatted_rows:
        print(format_row(row))


class ValidationGenerationMetricsCallback(TrainerCallback):
    def __init__(self, model, tokenizer, validation_dataset):
        self.model = model
        self.tokenizer = tokenizer
        self.validation_dataset = validation_dataset
        self.epoch_metrics = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics if metrics is not None else {}

        generated = calculate_validation_generation_metrics(
            self.model,
            self.tokenizer,
            self.validation_dataset,
        )

        row = {
            "epoch": state.epoch,
            "training_loss": latest_training_loss(state.log_history),
            "validation_loss": metrics.get("eval_loss", float("nan")),
            "category_accuracy": generated["category_accuracy"],
            "severity_accuracy": generated["severity_accuracy"],
            "joint_accuracy": generated["joint_accuracy"],
        }

        metrics["eval_category_accuracy"] = row["category_accuracy"]
        metrics["eval_severity_accuracy"] = row["severity_accuracy"]
        metrics["eval_joint_accuracy"] = row["joint_accuracy"]

        self.epoch_metrics.append(row)
        print_metrics_table(self.epoch_metrics)

        return control

In [17]:
SEED = 42
MODEL_DTYPE = torch.bfloat16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LEARNING_RATE, NUM_TRAIN_EPOCHS = 1e-4, 8
PER_DEVICE_TRAIN_BATCH_SIZE, PER_DEVICE_EVAL_BATCH_SIZE = 1, 1
GRADIENT_ACCUMULATION_STEPS, MAX_SEQUENCE_LENGTH = 8, 512
GENERATION_MAX_NEW_TOKENS, VALIDATION_GENERATION_BATCH_SIZE = 128, 4
ALLOWED_CATEGORIES = {'APP-01', 'AUTH-01', 'DB-01', 'DB-02', 'DEV-01', 'MAIL-01', 'MOB-01', 'NET-01', 'SEC-01', 'SRV-01'}
ALLOWED_SEVERITIES = {'P1', 'P2', 'P3', 'P4'}

set_seed(SEED)
raw_datasets = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VALIDATION_FILE)})
train_texts = validate_split('train', raw_datasets['train'])
validation_texts = validate_split('validation', raw_datasets['validation'])
if train_texts & validation_texts:
    raise ValueError('Training and validation splits share exact ticket text.')
sft_datasets = raw_datasets.map(to_prompt_completion, remove_columns=raw_datasets['train'].column_names)
print(f"Training records: {len(sft_datasets['train'])}")
print(f"Validation records: {len(sft_datasets['validation'])}")
print(f'Model dtype: {MODEL_DTYPE}')

Training records: 1000
Validation records: 200
Model dtype: torch.bfloat16


In [ ]:
# Run fix_system_environment.ipynb before this cell to extract the Python development headers.
PYTHON_HEADER_DIR = ROOT_DIR / "python311-devel" / "payload" / "usr" / "include" / "python3.11"
PYTHON_HEADER_FILE = PYTHON_HEADER_DIR / "Python.h"

if not PYTHON_HEADER_FILE.is_file():
    raise FileNotFoundError(
        f"Missing Python development header: {PYTHON_HEADER_FILE}. "
        "Run fix_system_environment.ipynb successfully before starting training."
    )

existing_c_include_path = os.environ.get("C_INCLUDE_PATH", "")
existing_cpath = os.environ.get("CPATH", "")
os.environ["C_INCLUDE_PATH"] = (
    f"{PYTHON_HEADER_DIR}:{existing_c_include_path}"
    if existing_c_include_path
    else str(PYTHON_HEADER_DIR)
)
os.environ["CPATH"] = (
    f"{PYTHON_HEADER_DIR}:{existing_cpath}" if existing_cpath else str(PYTHON_HEADER_DIR)
)

print(f"Python headers available in: {PYTHON_HEADER_DIR}")
print(f"C_INCLUDE_PATH: {os.environ['C_INCLUDE_PATH']}")
print(f"CPATH: {os.environ['CPATH']}")

In [18]:
# to bring the model on GPU
DEVICE = torch.device('cuda:0')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
if model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {model.config.model_type!r}.')
model.config.use_cache = False
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', 
                         target_modules=LORA_TARGET_MODULES)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)

training_arguments = SFTConfig(output_dir=str(TRAINING_OUTPUT_DIR), learning_rate=LEARNING_RATE, num_train_epochs=NUM_TRAIN_EPOCHS, 
                               per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, 
                               per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, 
                               gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, 
                               max_length=MAX_SEQUENCE_LENGTH, eval_strategy='epoch', save_strategy='epoch', 
                               logging_strategy='epoch', save_total_limit=1, load_best_model_at_end=True,
                               disable_tqdm=True,
                               metric_for_best_model='eval_joint_accuracy', greater_is_better=True, completion_only_loss=True, 
                               gradient_checkpointing=True, bf16=True, bf16_full_eval=True, dataloader_pin_memory=True, 
                               optim='adamw_torch', report_to='none', seed=SEED)
callback = ValidationGenerationMetricsCallback(model, tokenizer, raw_datasets['validation'])

trainer = SFTTrainer(model=model, args=training_arguments, 
                     train_dataset=sft_datasets['train'], eval_dataset=sft_datasets['validation'], 
                     processing_class=tokenizer, callbacks=[callback])
trainer.remove_callback(PrinterCallback)

floating_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.is_floating_point()}
trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
if floating_dtypes != {MODEL_DTYPE} or trainable_dtypes != {MODEL_DTYPE}:
    raise RuntimeError(f'Expected BF16 floating and trainable parameters, got {floating_dtypes} and {trainable_dtypes}.')
trainer.model.print_trainable_parameters()

result = trainer.train()

trainer.save_model(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)
print(f'Training loss: {result.training_loss:.4f}')
print(f'Adapter saved to: {ADAPTER_OUTPUT_DIR}')

trainable params: 8,716,288 || all params: 1,729,291,264 || trainable%: 0.5040


opc-request-id: csid4aa67ad3460b9f44df48ac5e2db8/ac8be224eded4e4cb595ee1ebdf534dd/30C1C2D5D35A485C9244E75B4DAA468B

Command ID aab31721-5e3f-4b6b-a765-99ae03e67869 failed with java.lang.RuntimeException: [Command aab31721-5e3f-4b6b-a765-99ae03e67869 in Context 52945d56-8107-4a98-869a-b80bae8d7781] failed with error: 
 ---------------------------------------------------------------------------CalledProcessError                        Traceback (most recent call last)Cell In[79], line 39
     36     raise RuntimeError(f'Expected BF16 floating and trainable parameters, got {floating_dtypes} and {trainable_dtypes}.')
     37 trainer.model.print_trainable_parameters()
---> 39 result = trainer.train()
     41 trainer.save_model(ADAPTER_OUTPUT_DIR)
     42 tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)
File /aidp/libraries/python/amd/transformers/trainer.py:2325, in Trainer.train(self, resume_from_checkpoint, trial, ignore_keys_for_eval, **kwargs)
   2323         hf_hub_utils.enable_progress_b

In [73]:

# the problem is that in OSS probably the file has not completely be saved and it is immediately used... 
checkpoint_dir = Path(trainer.state.best_model_checkpoint)
checkpoint_file = checkpoint_dir / "adapter_model.safetensors"

print("Checkpoint:", checkpoint_dir)
print("File exists:", checkpoint_file.exists())
print("Size:", checkpoint_file.stat().st_size if checkpoint_file.exists() else "N/A")

Checkpoint: /Volumes/fine_tuning/fine_tuning/vol_finetuning/models/qwen3-1.7b-ticket-classification-lora/checkpoint-375


File exists: True
Size: 17484288
